In [1]:
import numpy as np
import pandas as pd

In [5]:
import joblib

from sklearn.base import BaseEstimator, TransformerMixin

In [9]:
class RawFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = np.array(X.columns, dtype=object)
        return self

    def transform(self, X):
        df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)

        if 'PassengerId' in df.columns:
            df['Passengerno'] = df['PassengerId'].astype(str).str.split('_').str.get(1)

        if 'Cabin' in df.columns:
            cabin_split = df['Cabin'].astype(str).str.split('/', expand=True)
            if cabin_split.shape[1] == 3:
                df[['Deck', 'Num', 'Side']] = cabin_split
            else:
                df['Deck'], df['Num'], df['Side'] = np.nan, np.nan, np.nan

        bill_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
        existing_bill_cols = [c for c in bill_cols if c in df.columns]
        if existing_bill_cols:
            df[existing_bill_cols] = df[existing_bill_cols].apply(pd.to_numeric, errors='coerce')
            df['TotalBill'] = df[existing_bill_cols].sum(axis=1)

        if 'Age' in df.columns:
            df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
            bins = [0, 12, 19, 35, 59, 150]
            labels = ['Child', 'Teenager', 'Young Adult', 'Adult', 'Senior']
            df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True, include_lowest=True).astype(str)

        return df

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = getattr(self, "feature_names_in_", [])
        
        cols = list(input_features)
        if 'PassengerId' in cols and 'Passengerno' not in cols:
            cols.append('Passengerno')
        if 'Cabin' in cols:
            for new_col in ['Deck', 'Num', 'Side']:
                if new_col not in cols:
                    cols.append(new_col)
        
        bill_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
        if any(c in cols for c in bill_cols) and 'TotalBill' not in cols:
            cols.append('TotalBill')
        if 'Age' in cols and 'AgeGroup' not in cols:
            cols.append('AgeGroup')
            
        return np.array(cols, dtype=object)


def safe_log1p(X):
    X = np.asarray(X, dtype=np.float64)
    return np.log1p(np.nan_to_num(X))

In [13]:
preprocessor = joblib.load('../app//preprocessor_pipeline.joblib')
model = joblib.load('../app/best_model.joblib')

AttributeError: Can't get attribute 'StringExtractionTransformer' on <module '__main__'>

In [11]:
train = pd.read_csv('../data/raw/train.csv')

In [12]:
train_trans = preprocessor.transform(train)

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
